In [0]:
"""
Job 01 — Bronze: POS Real-Time Stream (Kafka → Delta)
======================================================
Reads the POS Kafka topic via Auto Loader in continuous mode and lands
raw, untyped records into the bronze Delta table.

Trigger  : Continuous (always-on streaming job)
Schedule : None — runs as a long-lived Databricks Streaming job
Output   : bronze/pos_transactions_stream   (append, partitioned by _batch_date)
"""

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    TimestampType, DateType, LongType
)

In [0]:
# ── Path configuration ───────────────────────────────────────────────────────
BRONZE    = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze"
CKPT_BASE = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze"

def bronze(f): return f"{BRONZE}/{f}"
def ckpt(f):   return f"{CKPT_BASE}/{f}"

# ── Kafka configuration ──────────────────────────────────────────────────────
KAFKA_BOOTSTRAP = "your-kafka-broker:9092"   # replace with your broker address
KAFKA_TOPIC     = "pos_transactions"          # replace with your topic name

# ── Schema — all strings at bronze (raw landing, no coercion) ────────────────
# The Kafka message value is expected to be a JSON string.
# We parse it and cast later at silver. Bronze stays faithful to source.
pos_stream_schema = StructType([
    StructField("transaction_id",  StringType(), False),
    StructField("store_id",        StringType(), True),
    StructField("product_id",      StringType(), True),
    StructField("customer_id",     StringType(), True),
    StructField("timestamp",       StringType(), True),
    StructField("quantity",        StringType(), True),
    StructField("unit_price",      StringType(), True),
    StructField("total_amount",    StringType(), True),
    StructField("payment_method",  StringType(), True),
    StructField("channel",         StringType(), True),
])

In [0]:
# ── Read from Kafka ──────────────────────────────────────────────────────────
kafka_raw = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
         .option("subscribe", KAFKA_TOPIC)
         .option("startingOffsets", "latest")          # use "earliest" for backfill
         .option("failOnDataLoss", "false")            # safe default for Kafka compaction
         .load()
)

# ── Parse JSON payload from Kafka value field ────────────────────────────────
# Kafka delivers: key (bytes), value (bytes), topic, partition, offset, timestamp
df_pos_stream = (
    kafka_raw
    .select(
        F.from_json(
            F.col("value").cast("string"),
            pos_stream_schema
        ).alias("data"),
        F.col("topic").alias("_kafka_topic"),
        F.col("partition").alias("_kafka_partition"),
        F.col("offset").alias("_kafka_offset"),
        F.col("timestamp").alias("_kafka_timestamp"),   # Kafka event timestamp
    )
    .select(
        "data.*",
        "_kafka_topic",
        "_kafka_partition",
        "_kafka_offset",
        "_kafka_timestamp",
    )
    # Standard bronze metadata columns
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_batch_date",          F.current_date())
    .withColumn("_source",              F.lit("pos_realtime_stream"))
)

In [0]:
# ── Write to Bronze Delta table (continuous, append) ─────────────────────────
(
    df_pos_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ckpt("pos_transactions_stream"))
        .trigger(continuous="1 second")              # continuous mode — low latency
        .partitionBy("_batch_date")
        .toTable("mini_project_team_a3t7.bronze.pos_transactions_stream")
        # alternatively: .start(bronze("pos_transactions_stream"))
)

print("Bronze streaming job started: bronze/pos_transactions_stream")
print(f"  Kafka topic     : {KAFKA_TOPIC}")
print(f"  Checkpoint path : {ckpt('pos_transactions_stream')}")